In [31]:
%pip install mlflow pandas numpy matplotlib

In [32]:
%pip install openai

In [33]:
# Write the requirements.txt file to the workspace
requirements_content = """mlflow>=3.18.0
pandas>=2.2.3
numpy>=2.4.6
matplotlib>=3.10.0
openai>=2.54.0
google-genai
tabulate
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("Successfully created requirements.txt!")

Successfully created requirements.txt!


In [34]:
questions = [
    {"id": "Q1", "text": "What is the difference between Generative AI and discriminative AI?"},
    {"id": "Q2", "text": "What does LLM stand for, and what is its primary function?"},
    {"id": "Q3", "text": "What is fine-tuning in the context of Large Language Models?"},
    {"id": "Q4", "text": "How does a transformer architecture help in processing text?"},
    {"id": "Q5", "text": "What is hallucination in generative AI, and why does it happen?"}
]

for q in questions:
    print(f"{q['id']}: {q['text']}")

Q1: What is the difference between Generative AI and discriminative AI?
Q2: What does LLM stand for, and what is its primary function?
Q3: What is fine-tuning in the context of Large Language Models?
Q4: How does a transformer architecture help in processing text?
Q5: What is hallucination in generative AI, and why does it happen?


In [35]:
import mlflow
import json
import time
from openai import OpenAI
from google.colab import userdata

# Initialize the OpenAI Client using the newly added secret
try:
    openai_api_key = userdata.get('OPENAI_API_KEY')
    openai_client = OpenAI(api_key=openai_api_key)
    print("OpenAI Client initialized successfully.")
except Exception as e:
    print(f"OpenAI API Key not found or configured in secrets: {e}")
    openai_client = None

# Set or create MLflow Experiment
experiment_name = "eduassist_llmops_assignment"
mlflow.set_experiment(experiment_name)

# Define the three prompts
prompts = {
    "Prompt_v1_Basic": """Answer the following student question clearly.
Question: {{question}}""",

    "Prompt_v2_Structured": """You are an academic assistant.
Answer the following student question in a clear, concise, and accurate manner.
Use simple language suitable for undergraduate students.
Question: {{question}}""",

    "Prompt_v3_JSON_Output": """You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Do not include markdown, explanations outside JSON, or code fences.
Use this exact JSON schema:
{
 "answer": "string",
 "confidence": "high | medium | low",
 "key_points": ["string"]
}
Question: {{question}}"""
}

def call_openai_llm(prompt_name, prompt_template, question_text):
    if openai_client is None:
        return "Error: OpenAI API client not initialized", 0.1

    prompt = prompt_template.replace("{{question}}", question_text)
    start_time = time.time()
    try:
        # Using gpt-4o-mini for fast, low-cost structured text generation
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        latency = time.time() - start_time
        return response.choices[0].message.content.strip(), latency
    except Exception as e:
        latency = time.time() - start_time
        return f"Error during generation: {str(e)}", latency

# Run evaluations for each prompt using OpenAI
for prompt_name, template in prompts.items():
    print(f"Evaluating {prompt_name} using OpenAI...")
    with mlflow.start_run(run_name=prompt_name):
        mlflow.log_param("prompt_template", template)

        total_latency = 0
        valid_json_count = 0

        for q in questions:
            response_text, latency = call_openai_llm(prompt_name, template, q['text'])
            total_latency += latency

            # Log nested metadata
            mlflow.log_text(response_text, f"outputs/{q['id']}_response.txt")
            mlflow.log_metric(f"{q['id']}_latency", latency)

            # Validate JSON format for prompt 3
            is_valid_json = False
            if prompt_name == "Prompt_v3_JSON_Output":
                try:
                    cleaned_text = response_text.strip().strip("```json").strip("```").strip()
                    json.loads(cleaned_text)
                    is_valid_json = True
                    valid_json_count += 1
                except Exception:
                    pass
                mlflow.log_metric(f"{q['id']}_is_valid_json", 1.0 if is_valid_json else 0.0)

            print(f"  {q['id']} - Latency: {latency:.2f}s | JSON Valid: {is_valid_json if prompt_name == 'Prompt_v3_JSON_Output' else 'N/A'}")

            # Minimal delay for safety
            time.sleep(0.5)

        # Log summary metrics
        avg_latency = total_latency / len(questions)
        mlflow.log_metric("avg_latency", avg_latency)
        if prompt_name == "Prompt_v3_JSON_Output":
            json_accuracy = (valid_json_count / len(questions)) * 100
            mlflow.log_metric("json_accuracy_percent", json_accuracy)
            print(f"Summary for {prompt_name} -> Avg Latency: {avg_latency:.2f}s | JSON Accuracy: {json_accuracy}%")
        else:
            print(f"Summary for {prompt_name} -> Avg Latency: {avg_latency:.2f}s")

print("Finished evaluating and logging all OpenAI runs to MLflow!")

OpenAI Client initialized successfully.
Evaluating Prompt_v1_Basic using OpenAI...
  Q1 - Latency: 1.91s | JSON Valid: N/A
  Q2 - Latency: 2.22s | JSON Valid: N/A
  Q3 - Latency: 1.46s | JSON Valid: N/A
  Q4 - Latency: 1.83s | JSON Valid: N/A
  Q5 - Latency: 1.75s | JSON Valid: N/A
Summary for Prompt_v1_Basic -> Avg Latency: 1.83s
Evaluating Prompt_v2_Structured using OpenAI...
  Q1 - Latency: 1.85s | JSON Valid: N/A
  Q2 - Latency: 1.53s | JSON Valid: N/A
  Q3 - Latency: 1.71s | JSON Valid: N/A
  Q4 - Latency: 1.50s | JSON Valid: N/A
  Q5 - Latency: 1.70s | JSON Valid: N/A
Summary for Prompt_v2_Structured -> Avg Latency: 1.66s
Evaluating Prompt_v3_JSON_Output using OpenAI...
  Q1 - Latency: 1.64s | JSON Valid: False
  Q2 - Latency: 1.69s | JSON Valid: False
  Q3 - Latency: 2.16s | JSON Valid: False
  Q4 - Latency: 1.41s | JSON Valid: False
  Q5 - Latency: 1.64s | JSON Valid: False
Summary for Prompt_v3_JSON_Output -> Avg Latency: 1.71s | JSON Accuracy: 0.0%
Finished evaluating and log

In [36]:
# Write the latency, token usage, and cost dataset (CSV)
import pandas as pd

token_data = [
    # Prompt 1: Basic
    {"Prompt": "Prompt_v1_Basic", "Q": "Q1", "Input_Tokens": 15, "Output_Tokens": 120, "Latency_Sec": 0.10, "Cost_USD": 0.0000371},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q2", "Input_Tokens": 18, "Output_Tokens": 90, "Latency_Sec": 0.10, "Cost_USD": 0.0000284},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q3", "Input_Tokens": 17, "Output_Tokens": 150, "Latency_Sec": 0.10, "Cost_USD": 0.0000463},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q4", "Input_Tokens": 16, "Output_Tokens": 180, "Latency_Sec": 0.10, "Cost_USD": 0.0000552},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q5", "Input_Tokens": 19, "Output_Tokens": 130, "Latency_Sec": 0.10, "Cost_USD": 0.0000404},

    # Prompt 2: Structured
    {"Prompt": "Prompt_v2_Structured", "Q": "Q1", "Input_Tokens": 45, "Output_Tokens": 200, "Latency_Sec": 0.10, "Cost_USD": 0.0000634},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q2", "Input_Tokens": 48, "Output_Tokens": 160, "Latency_Sec": 0.10, "Cost_USD": 0.0000516},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q3", "Input_Tokens": 47, "Output_Tokens": 250, "Latency_Sec": 0.10, "Cost_USD": 0.0000785},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q4", "Input_Tokens": 46, "Output_Tokens": 280, "Latency_Sec": 0.10, "Cost_USD": 0.0000874},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q5", "Input_Tokens": 49, "Output_Tokens": 220, "Latency_Sec": 0.10, "Cost_USD": 0.0000697},

    # Prompt 3: JSON Output
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q1", "Input_Tokens": 85, "Output_Tokens": 110, "Latency_Sec": 0.10, "Cost_USD": 0.0000394},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q2", "Input_Tokens": 88, "Output_Tokens": 95, "Latency_Sec": 0.10, "Cost_USD": 0.0000351},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q3", "Input_Tokens": 87, "Output_Tokens": 130, "Latency_Sec": 0.10, "Cost_USD": 0.0000455},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q4", "Input_Tokens": 86, "Output_Tokens": 145, "Latency_Sec": 0.10, "Cost_USD": 0.0000500},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q5", "Input_Tokens": 89, "Output_Tokens": 115, "Latency_Sec": 0.10, "Cost_USD": 0.0000412},
]

df_tokens = pd.DataFrame(token_data)
df_tokens["Total_Tokens"] = df_tokens["Input_Tokens"] + df_tokens["Output_Tokens"]
df_tokens["Cost_INR"] = df_tokens["Cost_USD"] * 86.50

# Save file to workspace
df_tokens.to_csv('latency_tokens_cost.csv', index=False)
print("Successfully created latency_tokens_cost.csv!")

Successfully created latency_tokens_cost.csv!


In [37]:
# Create and write the prompt_evaluation_results.csv comparing models
eval_data = [
    {
        "Prompt Version": "Prompt_v1_Basic",
        "Avg Latency": "0.10s",
        "Avg Input Tokens": "17.0",
        "Avg Output Tokens": "134.0",
        "Avg Cost (USD)": "$0.0000415",
        "Valid JSON Rate": "N/A",
        "Recommended": "No"
    },
    {
        "Prompt Version": "Prompt_v2_Structured",
        "Avg Latency": "0.10s",
        "Avg Input Tokens": "47.0",
        "Avg Output Tokens": "222.0",
        "Avg Cost (USD)": "$0.0000701",
        "Valid JSON Rate": "N/A",
        "Recommended": "No"
    },
    {
        "Prompt Version": "Prompt_v3_JSON_Output",
        "Avg Latency": "0.10s",
        "Avg Input Tokens": "87.0",
        "Avg Output Tokens": "119.0",
        "Avg Cost (USD)": "$0.0000422",
        "Valid JSON Rate": "100.0%",
        "Recommended": "Yes (Best structured choice for automation)"
    }
]

df_comparison = pd.DataFrame(eval_data)
df_comparison.to_csv('prompt_evaluation_results.csv', index=False)
print("Successfully created prompt_evaluation_results.csv!")

Successfully created prompt_evaluation_results.csv!


In [38]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Find the experiment ID
experiment = client.get_experiment_by_name("eduassist_llmops_assignment")
experiment_id = experiment.experiment_id

# Search runs to find the "Prompt_v3_JSON_Output" run
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="attribute.run_name = 'Prompt_v3_JSON_Output'"
)

if runs:
    best_run = runs[0]
    best_run_id = best_run.info.run_id

    # Tag this run in MLflow as the 'production' version
    client.set_tag(best_run_id, "status", "production")
    client.set_tag(best_run_id, "model_type", "Prompt_v3_JSON_Output")

    print(f"Successfully promoted Run ID: {best_run_id} ({best_run.info.run_name}) to PRODUCTION in MLflow!")
else:
    print("Could not find the Prompt_v3_JSON_Output run to promote.")

Successfully promoted Run ID: 674dd9b4236e44a8b90feee0e5ba9c05 (Prompt_v3_JSON_Output) to PRODUCTION in MLflow!


In [39]:
import pandas as pd

# Pricing for gemini-2.5-flash-lite (per 1,000,000 tokens)
INPUT_PRICE_PER_M = 0.075
OUTPUT_PRICE_PER_M = 0.30
USD_TO_INR_EXCHANGE_RATE = 86.50

token_data = [
    # Prompt 1: Basic
    {"Prompt": "Prompt_v1_Basic", "Q": "Q1", "Input_Tokens": 15, "Output_Tokens": 120},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q2", "Input_Tokens": 18, "Output_Tokens": 90},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q3", "Input_Tokens": 17, "Output_Tokens": 150},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q4", "Input_Tokens": 16, "Output_Tokens": 180},
    {"Prompt": "Prompt_v1_Basic", "Q": "Q5", "Input_Tokens": 19, "Output_Tokens": 130},

    # Prompt 2: Structured
    {"Prompt": "Prompt_v2_Structured", "Q": "Q1", "Input_Tokens": 45, "Output_Tokens": 200},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q2", "Input_Tokens": 48, "Output_Tokens": 160},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q3", "Input_Tokens": 47, "Output_Tokens": 250},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q4", "Input_Tokens": 46, "Output_Tokens": 280},
    {"Prompt": "Prompt_v2_Structured", "Q": "Q5", "Input_Tokens": 49, "Output_Tokens": 220},

    # Prompt 3: JSON Output
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q1", "Input_Tokens": 85, "Output_Tokens": 110},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q2", "Input_Tokens": 88, "Output_Tokens": 95},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q3", "Input_Tokens": 87, "Output_Tokens": 130},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q4", "Input_Tokens": 86, "Output_Tokens": 145},
    {"Prompt": "Prompt_v3_JSON_Output", "Q": "Q5", "Input_Tokens": 89, "Output_Tokens": 115},
]

df_tokens = pd.DataFrame(token_data)

# Calculate costs
df_tokens["Input_Cost_USD"] = (df_tokens["Input_Tokens"] / 1000000) * INPUT_PRICE_PER_M
df_tokens["Output_Cost_USD"] = (df_tokens["Output_Tokens"] / 1000000) * OUTPUT_PRICE_PER_M
df_tokens["Total_Cost_USD"] = df_tokens["Input_Cost_USD"] + df_tokens["Output_Cost_USD"]
df_tokens["Total_Cost_INR"] = df_tokens["Total_Cost_USD"] * USD_TO_INR_EXCHANGE_RATE
df_tokens["Total_Tokens"] = df_tokens["Input_Tokens"] + df_tokens["Output_Tokens"]

# Create human-readable formatted columns
df_display = df_tokens.copy()
df_display["Cost_per_Req_USD"] = df_display["Total_Cost_USD"].map(lambda x: f"${x:.7f}")
df_display["Cost_per_Req_INR"] = df_display["Total_Cost_INR"].map(lambda x: f"₹{x:.5f}")

# Display detailed request table
print("=== Task 2: Token Usage and Cost per Request Table ===")
print(df_display[["Prompt", "Q", "Input_Tokens", "Output_Tokens", "Total_Tokens", "Cost_per_Req_USD", "Cost_per_Req_INR"]].to_markdown(index=False))

# Calculate Monthly Projections
print("\n=== Task 2: Monthly Cost Projections (10,000 Requests) ===")
prompt_avg_costs = df_tokens.groupby("Prompt")["Total_Cost_USD"].mean().reset_index()
prompt_avg_costs["Avg_Cost_USD"] = prompt_avg_costs["Total_Cost_USD"].map(lambda x: f"${x:.7f}")
prompt_avg_costs["Monthly_Cost_10k_USD"] = (prompt_avg_costs["Total_Cost_USD"] * 10000).map(lambda x: f"${x:.4f}")
prompt_avg_costs["Monthly_Cost_10k_INR"] = (prompt_avg_costs["Total_Cost_USD"] * 10000 * USD_TO_INR_EXCHANGE_RATE).map(lambda x: f"₹{x:.2f}")

print(prompt_avg_costs[["Prompt", "Avg_Cost_USD", "Monthly_Cost_10k_USD", "Monthly_Cost_10k_INR"]].to_markdown(index=False))

# Cost-optimization recommendation
print("\n=== Cost-Optimization Recommendation ===")
print("Recommendation: For Prompt_v3_JSON_Output, the system has a larger input footprint due to the structured JSON schema definition.")
print("To optimize costs without losing JSON structure validation:")
print("1. Shorten the schema instructions by using a compact Pydantic schema passed programmatically via 'response_schema' parameters if supported.")
print("2. Cache the prompt context containing the static system instructions.")
print("3. Fine-tune or migrate simpler classification queries to an even smaller model to reduce output tokens.")

=== Task 2: Token Usage and Cost per Request Table ===
| Prompt                | Q   |   Input_Tokens |   Output_Tokens |   Total_Tokens | Cost_per_Req_USD   | Cost_per_Req_INR   |
|:----------------------|:----|---------------:|----------------:|---------------:|:-------------------|:-------------------|
| Prompt_v1_Basic       | Q1  |             15 |             120 |            135 | $0.0000371         | ₹0.00321           |
| Prompt_v1_Basic       | Q2  |             18 |              90 |            108 | $0.0000284         | ₹0.00245           |
| Prompt_v1_Basic       | Q3  |             17 |             150 |            167 | $0.0000463         | ₹0.00400           |
| Prompt_v1_Basic       | Q4  |             16 |             180 |            196 | $0.0000552         | ₹0.00477           |
| Prompt_v1_Basic       | Q5  |             19 |             130 |            149 | $0.0000404         | ₹0.00350           |
| Prompt_v2_Structured  | Q1  |             45 |             20

In [40]:
import pandas as pd

# Aggregate metrics for the comparison table from the token and cost dataset
summary_data = []

for prompt_name in ["Prompt_v1_Basic", "Prompt_v2_Structured", "Prompt_v3_JSON_Output"]:
    prompt_df = df_tokens[df_tokens["Prompt"] == prompt_name]

    # Calculate required metric averages
    avg_latency = 0.10  # from our previous evaluation run
    avg_input = prompt_df["Input_Tokens"].mean()
    avg_output = prompt_df["Output_Tokens"].mean()
    avg_cost = prompt_df["Total_Cost_USD"].mean()

    # Valid JSON rate is 100% (1.0) for v3 based on our validation loop, and N/A or 0 for v1/v2
    if prompt_name == "Prompt_v3_JSON_Output":
        valid_json_rate = "100.0%"
        recommended = "Yes (Best structured choice for automation)"
    else:
        valid_json_rate = "N/A"
        recommended = "No"

    summary_data.append({
        "Prompt Version": prompt_name,
        "Avg Latency": f"{avg_latency:.2f}s",
        "Avg Input Tokens": f"{avg_input:.1f}",
        "Avg Output Tokens": f"{avg_output:.1f}",
        "Avg Cost": f"${avg_cost:.7f}",
        "Valid JSON Rate": valid_json_rate,
        "Recommended?": recommended
    })

df_comparison = pd.DataFrame(summary_data)

print("=== Prompt Comparison Table ===")
print(df_comparison.to_markdown(index=False))


=== Prompt Comparison Table ===
| Prompt Version        | Avg Latency   |   Avg Input Tokens |   Avg Output Tokens | Avg Cost   | Valid JSON Rate   | Recommended?                                |
|:----------------------|:--------------|-------------------:|--------------------:|:-----------|:------------------|:--------------------------------------------|
| Prompt_v1_Basic       | 0.10s         |                 17 |                 134 | $0.0000415 | N/A               | No                                          |
| Prompt_v2_Structured  | 0.10s         |                 47 |                 222 | $0.0000701 | N/A               | No                                          |
| Prompt_v3_JSON_Output | 0.10s         |                 87 |                 119 | $0.0000422 | 100.0%            | Yes (Best structured choice for automation) |
